# Phase 1: 数据探索 & 特征工程

本 notebook 覆盖 Phase 1 的四个任务：
- **Task A**: 停电数据探索（目标变量）
- **Task B**: 天气特征探索 & 特征筛选
- **Task C**: 通用特征构造
- **Task D**: Baseline 复现 & 评估框架

每个人负责自己的部分，但建议大家都跑一遍，了解全貌。

In [ ]:
# ============== 安装 & 导入 ==============
# Colab 用户取消下面的注释
# !pip install netCDF4 xarray

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

# ====== 中文字体配置 ======
import matplotlib
import platform
if platform.system() == 'Darwin':  # macOS
    matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti']
elif platform.system() == 'Linux':  # Linux / Colab
    matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC', 'SimHei']
else:  # Windows
    matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 数据路径 — 确保 train.nc 已经放到 data/ 文件夹
DATA_DIR = "data/"
ds = xr.open_dataset(os.path.join(DATA_DIR, "train.nc"))

# 基本信息提取
timestamps = pd.to_datetime(ds.timestamp.values)
locations = list(ds.location.values)
weather_features = list(ds.feature.values) if 'feature' in ds.dims else []

# 核心数据矩阵
outage = ds.out.transpose("timestamp", "location").values.astype(float)      # (T, L)
tracked = ds.tracked.transpose("timestamp", "location").values.astype(float)  # (T, L)
weather = ds.weather.transpose("timestamp", "location", "feature").values.astype(float)  # (T, L, F)

T, L = outage.shape
F = len(weather_features)

print(f"时间范围: {timestamps.min()} ~ {timestamps.max()}")
print(f"时间步数: {T} (小时)")
print(f"县的数量: {L}")
print(f"天气特征数: {F}")
print(f"停电数据 shape: {outage.shape}")
print(f"天气数据 shape: {weather.shape}")

---
# Task A: 停电数据探索

**目标**：搞清楚停电数据长什么样——分布、哪些县严重、有没有时间规律。

## A1: 停电数据的整体分布

先看看 `out` 这个变量长什么样。剧透：绝大多数时间没停电（=0），偶尔有极端大值。

In [ ]:
# A1: 停电数据的基本统计
flat_out = outage.flatten()
print("=== 停电数据基本统计 ===")
print(f"总观测数: {len(flat_out):,}")
print(f"零值占比: {(flat_out == 0).mean():.1%}")
print(f"均值: {flat_out.mean():.2f}")
print(f"中位数: {np.median(flat_out):.2f}")
print(f"标准差: {flat_out.std():.2f}")
print(f"最大值: {flat_out.max():.0f}")
print(f"99%分位: {np.percentile(flat_out, 99):.0f}")
print(f"99.9%分位: {np.percentile(flat_out, 99.9):.0f}")

# 直方图：排除零值看分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：全部数据
axes[0].hist(flat_out, bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title("停电数分布 (全部数据)")
axes[0].set_xlabel("停电户数")
axes[0].set_ylabel("频次")
axes[0].set_yscale('log')

# 右图：只看非零值
nonzero = flat_out[flat_out > 0]
axes[1].hist(nonzero, bins=100, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title(f"停电数分布 (仅非零值, n={len(nonzero):,})")
axes[1].set_xlabel("停电户数")
axes[1].set_ylabel("频次")
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"\n结论: 数据极度右偏，{(flat_out==0).mean():.0%} 的观测值为0，极端值可达 {flat_out.max():.0f}。")
print("RMSE 对大值非常敏感，所以准确预测极端停电事件是拿分关键。")

## A2: 停电数据分层 — 零值 / 日常 / 严重 / 极端

A1 告诉我们数据极度稀疏且右偏。直接取平均会被极端值主导，掩盖日常模式。
所以先把数据分层，后续分析都基于这个分层框架。

**分层逻辑**（基于非零值分位数）：
- **零值**: out = 0（占绝大多数）
- **日常停电**: 0 < out ≤ P75（非零值）— 小规模、零散停电
- **严重停电**: P75 < out ≤ P99 — 区域性事件
- **极端停电**: out > P99 — 大规模风暴级停电（RMSE 的主要来源）

In [ ]:
# A2: 停电数据分层
flat_out = outage.flatten()
nonzero = flat_out[flat_out > 0]

# 基于非零值的分位数定义阈值
p75 = np.percentile(nonzero, 75)
p99 = np.percentile(nonzero, 99)

print("=== 分层阈值 ===")
print(f"  非零值 P75 = {p75:.0f} (日常 / 严重 分界)")
print(f"  非零值 P99 = {p99:.0f} (严重 / 极端 分界)")

# 给每个观测打标签
def classify_outage(val):
    if val == 0: return '零值'
    elif val <= p75: return '日常'
    elif val <= p99: return '严重'
    else: return '极端'

labels = np.array([classify_outage(v) for v in flat_out])
regime_order = ['零值', '日常', '严重', '极端']

# --- 统计表 ---
print("\n=== 各层统计 ===")
print(f"{'层级':>6s} | {'观测数':>10s} | {'占比':>7s} | {'均值':>10s} | {'中位数':>8s} | {'最大值':>10s} | {'占总停电量':>10s}")
print("-" * 80)
total_sum = flat_out.sum()
for regime in regime_order:
    mask = labels == regime
    vals = flat_out[mask]
    n = len(vals)
    pct = n / len(flat_out) * 100
    mean_v = vals.mean() if n > 0 else 0
    med_v = np.median(vals) if n > 0 else 0
    max_v = vals.max() if n > 0 else 0
    sum_pct = vals.sum() / total_sum * 100 if total_sum > 0 else 0
    print(f"{'':>2s}{regime:>4s} | {n:>10,d} | {pct:>6.1f}% | {mean_v:>10.1f} | {med_v:>8.1f} | {max_v:>10.0f} | {sum_pct:>9.1f}%")

# --- 可视化 ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'零值': '#bdbdbd', '日常': '#4CAF50', '严重': '#FF9800', '极端': '#F44336'}

# 1) 各层观测数量占比（饼图）
counts = [np.sum(labels == r) for r in regime_order]
axes[0].pie(counts, labels=regime_order, colors=[colors[r] for r in regime_order],
            autopct='%1.1f%%', startangle=90, explode=[0, 0, 0.05, 0.1])
axes[0].set_title('各层观测数量占比')

# 2) 各层对总停电量的贡献（饼图）
sums = [flat_out[labels == r].sum() for r in regime_order]
axes[1].pie(sums, labels=regime_order, colors=[colors[r] for r in regime_order],
            autopct='%1.1f%%', startangle=90, explode=[0, 0, 0.05, 0.1])
axes[1].set_title('各层对总停电量的贡献')

# 3) 非零各层的分布 (box plot)
data_by_regime = [flat_out[labels == r] for r in ['日常', '严重', '极端']]
bp = axes[2].boxplot(data_by_regime, labels=['日常', '严重', '极端'], patch_artist=True, showfliers=False)
for patch, r in zip(bp['boxes'], ['日常', '严重', '极端']):
    patch.set_facecolor(colors[r])
axes[2].set_ylabel('停电户数')
axes[2].set_title('非零各层的分布 (不含离群值)')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"\n关键发现: 极端层虽然只占观测的很小比例，但贡献了总停电量的大部分。")
print(f"  → RMSE 优化的重点应该放在准确预测这些极端事件上。")
print(f"  → 后续分析将分层进行，避免极端值掩盖日常模式。")

## A3: 各县停电分析 — 热力图 + 地理映射

用两种方式看县级停电分布：
1. **时间-县 热力图**：直观看哪些县在什么时候停电最严重
2. **密歇根州地理热力图**：基于 FIPS 编码的 choropleth 地图

In [ ]:
# A3: 各县停电分析

# ====== 3.1 各县多维度统计 ======
county_stats = pd.DataFrame({
    'fips': locations,
    'total_outage': outage.sum(axis=0),
    'mean_outage': outage.mean(axis=0),
    'max_outage': outage.max(axis=0),
    'mean_tracked': tracked.mean(axis=0),
    'nonzero_hours': (outage > 0).sum(axis=0),           # 有停电的小时数
    'severe_hours': (outage > p75).sum(axis=0),           # 严重停电小时数
    'extreme_hours': (outage > p99).sum(axis=0),          # 极端停电小时数
})
county_stats['outage_rate'] = county_stats['mean_outage'] / county_stats['mean_tracked'].replace(0, np.nan)
county_stats['nonzero_pct'] = county_stats['nonzero_hours'] / T * 100

# 按总停电量排序
county_stats = county_stats.sort_values('total_outage', ascending=False).reset_index(drop=True)

print("=== 各县停电统计 Top 15 ===")
print(county_stats.head(15)[['fips', 'total_outage', 'max_outage', 'nonzero_pct',
                              'severe_hours', 'extreme_hours', 'mean_tracked']].to_string(index=False))

# ====== 3.2 时间-县 热力图 ======
# 按总停电量排序县，按天聚合时间（小时太密看不清）
daily_outage = pd.DataFrame(outage, index=timestamps).resample('D').sum().values  # (Days, L)
sorted_county_idx = county_stats['fips'].apply(lambda x: locations.index(x)).values

fig, ax = plt.subplots(figsize=(18, 10))
# 用 log(1+x) 避免零值和极端值问题
im = ax.imshow(np.log1p(daily_outage[:, sorted_county_idx].T),
               aspect='auto', cmap='YlOrRd', interpolation='nearest')
cbar = plt.colorbar(im, ax=ax, label='log(1 + 日停电总数)')

# 标注
n_days = daily_outage.shape[0]
tick_step = max(n_days // 10, 1)
day_labels = pd.date_range(timestamps[0], periods=n_days, freq='D')
ax.set_xticks(range(0, n_days, tick_step))
ax.set_xticklabels([d.strftime('%m/%d') for d in day_labels[::tick_step]], rotation=45)
ax.set_yticks(range(0, len(locations), 5))
ax.set_yticklabels(county_stats['fips'].values[::5], fontsize=8)
ax.set_xlabel('日期')
ax.set_ylabel('县 (FIPS, 按总停电量排序)')
ax.set_title('各县每日停电热力图 — 颜色越深停电越严重')
plt.tight_layout()
plt.show()

print("热力图观察: 可以清晰看到停电事件在时间和空间上的集中性 — 少数风暴日+少数县贡献了大部分停电。")

# ====== 3.3 密歇根州地理热力图 ======
try:
    import plotly.express as px
    from urllib.request import urlopen
    import json

    with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
        counties_geojson = json.load(response)

    # 准备数据 — 多个指标分别画
    geo_df = county_stats[['fips', 'total_outage', 'max_outage', 'extreme_hours', 'outage_rate']].copy()
    geo_df['log_total'] = np.log1p(geo_df['total_outage'])

    fig = px.choropleth(
        geo_df, geojson=counties_geojson, locations='fips',
        color='log_total', color_continuous_scale='YlOrRd',
        scope='usa', title='密歇根州各县停电总量 (log scale)',
        labels={'log_total': 'log(1+总停电)'}
    )
    fig.update_geos(fitbounds="locations")
    fig.update_layout(width=700, height=500, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

    # 极端停电小时数地图
    fig2 = px.choropleth(
        geo_df, geojson=counties_geojson, locations='fips',
        color='extreme_hours', color_continuous_scale='Reds',
        scope='usa', title='密歇根州各县极端停电小时数',
        labels={'extreme_hours': '极端停电小时数'}
    )
    fig2.update_geos(fitbounds="locations")
    fig2.update_layout(width=700, height=500, margin=dict(l=0, r=0, t=40, b=0))
    fig2.show()

    print("地理热力图: 停电严重的县在地理上是否有聚集性？是否集中在沿海/特定气候区？")
except Exception as e:
    print(f"plotly 地理图加载失败 ({e})，跳过。上面的热力图已经包含了核心信息。")

## A4: 分层时间模式分析 — 风暴时段 vs 日常时段

A1 中按小时聚合看不出模式，是因为极端风暴事件随机发生，把日常规律淹没了。
解决方法：先识别"风暴时段"，然后分开分析。

**风暴时段定义**：全州小时总停电 > P95 的连续时段（及前后 6 小时缓冲区）

In [ ]:
# A4: 分层时间模式分析

total_outage_by_time = outage.sum(axis=1)  # (T,) 每小时全州总停电

# ====== 4.1 识别风暴时段 ======
storm_threshold = np.percentile(total_outage_by_time, 95)
is_storm_raw = total_outage_by_time > storm_threshold

# 加 6 小时缓冲区（风暴前后也算风暴时段）
buffer_hours = 6
is_storm = is_storm_raw.copy()
for i in range(T):
    if is_storm_raw[i]:
        start = max(0, i - buffer_hours)
        end = min(T, i + buffer_hours + 1)
        is_storm[start:end] = True

is_quiet = ~is_storm

n_storm = is_storm.sum()
n_quiet = is_quiet.sum()
print(f"=== 时段划分 ===")
print(f"  风暴时段: {n_storm} 小时 ({n_storm/T*100:.1f}%)")
print(f"  日常时段: {n_quiet} 小时 ({n_quiet/T*100:.1f}%)")
print(f"  风暴阈值 (全州小时总停电 P95): {storm_threshold:.0f}")
print(f"  风暴时段停电占比: {total_outage_by_time[is_storm].sum()/total_outage_by_time.sum()*100:.1f}%")

# ====== 4.2 全时序图 + 风暴时段标注 ======
fig, axes = plt.subplots(4, 1, figsize=(18, 18))

# 图1: 全时序 + 风暴区间高亮
ax = axes[0]
ax.plot(timestamps, total_outage_by_time, linewidth=0.6, color='black', alpha=0.7)
# 标记风暴区间
storm_starts = []
storm_ends = []
in_storm = False
for i in range(T):
    if is_storm[i] and not in_storm:
        storm_starts.append(i)
        in_storm = True
    elif not is_storm[i] and in_storm:
        storm_ends.append(i)
        in_storm = False
if in_storm:
    storm_ends.append(T - 1)

for s, e in zip(storm_starts, storm_ends):
    ax.axvspan(timestamps[s], timestamps[e], alpha=0.2, color='red')

ax.set_title(f'全州每小时总停电 (红色区域 = 风暴时段, 共 {len(storm_starts)} 次)')
ax.set_ylabel('停电户数')

# 标出 top 5 峰值
peak_idx = np.argsort(total_outage_by_time)[-5:]
for idx in peak_idx:
    ax.annotate(f"{timestamps[idx].strftime('%m/%d %Hh')}\n{total_outage_by_time[idx]:,.0f}",
                xy=(timestamps[idx], total_outage_by_time[idx]),
                fontsize=8, color='red', ha='center')

# ====== 4.3 分层日内模式 (小时) ======
ax = axes[1]
hours = timestamps.hour

# 日常时段
quiet_hourly = pd.Series(total_outage_by_time[is_quiet]).groupby(hours[is_quiet]).agg(['mean', 'std'])
# 风暴时段
storm_hourly = pd.Series(total_outage_by_time[is_storm]).groupby(hours[is_storm]).agg(['mean', 'std'])

x = np.arange(24)
width = 0.35
bars1 = ax.bar(x - width/2, quiet_hourly['mean'], width, label='日常时段', color='#4CAF50', alpha=0.8)
bars2 = ax.bar(x + width/2, storm_hourly['mean'], width, label='风暴时段', color='#F44336', alpha=0.8)
ax.set_xlabel('小时 (0-23)')
ax.set_ylabel('平均停电户数')
ax.set_title('日内停电模式 — 日常 vs 风暴 (注意 y 轴差异巨大)')
ax.set_xticks(range(24))
ax.legend()

# 加第二个 y 轴显示日常时段（因为量级差太大）
ax2 = ax.twinx()
ax2.plot(x, quiet_hourly['mean'], 'o-', color='darkgreen', linewidth=2, markersize=4, label='日常时段 (右轴)')
ax2.set_ylabel('日常时段平均停电户数', color='darkgreen')
ax2.tick_params(axis='y', labelcolor='darkgreen')

# ====== 4.4 日常时段细看：存在日内周期性吗？ ======
ax = axes[2]
# 只看日常时段，计算每小时的均值和置信区间
quiet_by_hour = []
for h in range(24):
    mask = is_quiet & (hours == h)
    quiet_by_hour.append(total_outage_by_time[mask])

bp = ax.boxplot(quiet_by_hour, positions=range(24), patch_artist=True, showfliers=False,
                boxprops=dict(facecolor='#4CAF50', alpha=0.6))
ax.set_xlabel('小时 (0-23)')
ax.set_ylabel('停电户数')
ax.set_title('日常时段的日内模式 (箱线图, 不含离群值)')
ax.set_xticks(range(24))

# ====== 4.5 星期模式 ======
ax = axes[3]
dow_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
dow = timestamps.dayofweek

# 日常
quiet_dow = pd.Series(total_outage_by_time[is_quiet]).groupby(dow[is_quiet]).mean()
# 风暴
storm_dow = pd.Series(total_outage_by_time[is_storm]).groupby(dow[is_storm]).mean()

x = np.arange(7)
ax.bar(x - width/2, quiet_dow.reindex(range(7), fill_value=0), width,
       label='日常时段', color='#4CAF50', alpha=0.8)
ax.bar(x + width/2, storm_dow.reindex(range(7), fill_value=0), width,
       label='风暴时段', color='#F44336', alpha=0.8)
ax.set_xlabel('星期')
ax.set_ylabel('平均停电户数')
ax.set_title('星期模式 — 日常 vs 风暴')
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.legend()

plt.tight_layout()
plt.show()

# ====== 4.6 风暴事件清单 ======
print(f"\n=== 识别到的 {len(storm_starts)} 次风暴事件 ===")
print(f"{'编号':>4s} | {'起始时间':>16s} | {'结束时间':>16s} | {'持续(h)':>7s} | {'峰值停电':>10s} | {'累计停电':>12s}")
print("-" * 80)
for i, (s, e) in enumerate(zip(storm_starts, storm_ends)):
    duration = e - s
    peak = total_outage_by_time[s:e].max()
    total = total_outage_by_time[s:e].sum()
    print(f"{i+1:>4d} | {timestamps[s].strftime('%Y-%m-%d %H:%M'):>16s} | "
          f"{timestamps[min(e, T-1)].strftime('%Y-%m-%d %H:%M'):>16s} | "
          f"{duration:>7d} | {peak:>10,.0f} | {total:>12,.0f}")

print(f"\n结论:")
print(f"  1. 日常时段可能存在轻微的日内周期（午后略高？），但量级很小")
print(f"  2. 风暴时段的停电量级比日常高出几个数量级，是完全不同的数据生成过程")
print(f"  3. 建模策略建议: 可以考虑分别建模日常/风暴，或用天气特征让模型自动学习两种模式")

---
# Task B: 天气特征探索 & 特征筛选

**目标**：从 109 个天气变量里找出和停电最相关的，删掉冗余的，缩减到 15-25 个。

## B1: 天气特征与停电的相关性排名

把每个天气变量的全部数据展平，算和停电的 Pearson 相关系数，排序找 top 20。

In [ ]:
# B1: 各天气变量与停电的相关性
flat_out = outage.flatten()
correlations = {}

for fi, feat_name in enumerate(weather_features):
    flat_w = weather[:, :, fi].flatten()
    # 去掉 NaN
    mask = ~(np.isnan(flat_out) | np.isnan(flat_w))
    if mask.sum() > 100:
        correlations[feat_name] = np.corrcoef(flat_out[mask], flat_w[mask])[0, 1]

corr_series = pd.Series(correlations).sort_values(key=abs, ascending=False)

# 画 top 30
fig, ax = plt.subplots(figsize=(10, 10))
top30 = corr_series.head(30)
colors = ['coral' if v > 0 else 'steelblue' for v in top30.values]
ax.barh(range(len(top30)), top30.values, color=colors)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30.index)
ax.set_xlabel('Pearson 相关系数')
ax.set_title('天气特征与停电的相关性 (Top 30)')
ax.invert_yaxis()
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("=== Top 20 相关特征 ===")
for name, corr in corr_series.head(20).items():
    print(f"  {name:15s}: {corr:+.4f}")

print(f"\n=== 相关性很弱的特征 (|r| < 0.01, 候选删除) ===")
weak = corr_series[corr_series.abs() < 0.01]
print(f"  共 {len(weak)} 个: {list(weak.index)}")

## B2: 特征之间的共线性分析

如果两个天气变量相关性 > 0.9，说明它们几乎是同一个东西，留一个就行。

In [ ]:
# B2: Top 特征之间的共线性热力图
# 只看与停电相关性 top 25 的特征，避免热力图太大
top25_names = list(corr_series.head(25).index)
top25_indices = [weather_features.index(n) for n in top25_names]

# 用全局数据算特征间相关矩阵
weather_flat = weather.reshape(-1, F)  # (T*L, F)
top25_data = weather_flat[:, top25_indices]
feat_corr_matrix = np.corrcoef(top25_data.T)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(feat_corr_matrix, xticklabels=top25_names, yticklabels=top25_names,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=False, ax=ax)
ax.set_title("Top 25 天气特征之间的相关性矩阵")
plt.tight_layout()
plt.show()

# 找出高度共线性的特征对 (|r| > 0.9)
print("=== 高度共线性的特征对 (|r| > 0.9) ===")
print("可以考虑每对只保留一个（保留与停电相关性更强的那个）\n")
seen = set()
for i in range(len(top25_names)):
    for j in range(i+1, len(top25_names)):
        r = feat_corr_matrix[i, j]
        if abs(r) > 0.9:
            pair = f"  {top25_names[i]:15s} <-> {top25_names[j]:15s}: r={r:.3f}"
            print(pair)

## B3: 关键天气变量 vs 停电的时序对比

选几个最相关的天气变量，看它们在极端停电事件前后的变化。

In [ ]:
# B3: 关键天气变量 vs 全州停电的时序对比
# 选前 6 个最相关的特征
top6_feats = list(corr_series.head(6).index)

fig, axes = plt.subplots(len(top6_feats) + 1, 1, figsize=(16, 3 * (len(top6_feats) + 1)), sharex=True)

# 第一行：停电总数
axes[0].plot(timestamps, total_outage_by_time, color='black', linewidth=0.8)
axes[0].set_ylabel("停电总数")
axes[0].set_title("全州停电 vs 关键天气变量")

# 后续行：各天气变量（全县平均）
for i, feat_name in enumerate(top6_feats):
    fi = weather_features.index(feat_name)
    feat_avg = weather[:, :, fi].mean(axis=1)  # 全县平均
    corr_val = corr_series[feat_name]
    axes[i+1].plot(timestamps, feat_avg, linewidth=0.8, color='steelblue')
    axes[i+1].set_ylabel(feat_name)
    axes[i+1].set_title(f"{feat_name} (r={corr_val:+.3f})", loc='left', fontsize=10)

axes[-1].set_xlabel("时间")
plt.tight_layout()
plt.show()

## B4: 缺失值检查

In [ ]:
# B4: 缺失值检查
print("=== 缺失值统计 ===")
print(f"  out 缺失: {np.isnan(outage).sum()} / {outage.size}")
print(f"  tracked 缺失: {np.isnan(tracked).sum()} / {tracked.size}")

weather_nan_per_feat = np.isnan(weather.reshape(-1, F)).sum(axis=0)
nan_df = pd.DataFrame({'feature': weather_features, 'nan_count': weather_nan_per_feat})
nan_df['nan_pct'] = nan_df['nan_count'] / (T * L) * 100
nan_with_missing = nan_df[nan_df['nan_count'] > 0].sort_values('nan_count', ascending=False)

if len(nan_with_missing) > 0:
    print(f"\n  有缺失值的天气特征 ({len(nan_with_missing)} 个):")
    print(nan_with_missing.to_string(index=False))
else:
    print("\n  所有天气特征均无缺失值")

---
# Task C: 通用特征构造

**目标**：构造时间特征、滞后特征、滑动窗口统计，封装成函数大家共享。

这些特征无论后续用什么模型（树模型、LSTM、SARIMAX）都有用。

In [ ]:
# Task C: 通用特征构造函数
# 注意：所有特征只用过去的信息，不能用未来数据（防止数据泄漏）

def build_features(ds, selected_weather_features=None):
    """
    从 xarray dataset 构造特征 DataFrame。
    
    参数:
        ds: xarray Dataset，包含 out, tracked, weather, timestamp, location
        selected_weather_features: 筛选后的天气特征名列表 (None = 用全部)
    
    返回:
        df: DataFrame，每行是一个 (timestamp, location) 的观测，包含所有构造的特征
    """
    timestamps = pd.to_datetime(ds.timestamp.values)
    locations = list(ds.location.values)
    all_features = list(ds.feature.values)
    
    out = ds.out.transpose("timestamp", "location").values.astype(float)
    trk = ds.tracked.transpose("timestamp", "location").values.astype(float)
    w = ds.weather.transpose("timestamp", "location", "feature").values.astype(float)
    
    T, L = out.shape
    records = []
    
    for li in range(L):
        loc = locations[li]
        df_loc = pd.DataFrame({
            'timestamp': timestamps,
            'location': loc,
            'out': out[:, li],
            'tracked': trk[:, li],
        })
        
        # ====== 1. 时间特征 ======
        df_loc['hour'] = timestamps.hour
        df_loc['day_of_week'] = timestamps.dayofweek
        df_loc['month'] = timestamps.month
        # sin/cos 编码（让 23点和0点相近）
        df_loc['hour_sin'] = np.sin(2 * np.pi * df_loc['hour'] / 24)
        df_loc['hour_cos'] = np.cos(2 * np.pi * df_loc['hour'] / 24)
        df_loc['dow_sin'] = np.sin(2 * np.pi * df_loc['day_of_week'] / 7)
        df_loc['dow_cos'] = np.cos(2 * np.pi * df_loc['day_of_week'] / 7)
        
        # ====== 2. 停电滞后特征 ======
        for lag in [1, 3, 6, 12, 24]:
            df_loc[f'out_lag_{lag}h'] = df_loc['out'].shift(lag)
        
        # ====== 3. 停电滑动窗口统计 ======
        for window in [6, 12, 24]:
            rolling = df_loc['out'].shift(1).rolling(window, min_periods=1)
            df_loc[f'out_roll_mean_{window}h'] = rolling.mean()
            df_loc[f'out_roll_max_{window}h'] = rolling.max()
            df_loc[f'out_roll_std_{window}h'] = rolling.std().fillna(0)
        
        # ====== 4. 天气特征（筛选后的）======
        if selected_weather_features is not None:
            feat_indices = [all_features.index(f) for f in selected_weather_features]
            feat_names = selected_weather_features
        else:
            feat_indices = list(range(len(all_features)))
            feat_names = all_features
        
        for fi, fname in zip(feat_indices, feat_names):
            df_loc[f'w_{fname}'] = w[:, li, fi]
        
        # ====== 5. 关键天气的滑动统计（可选，只对 top 特征做）======
        # 这里先跳过，在确定 top 特征后再补充
        
        records.append(df_loc)
    
    df = pd.concat(records, ignore_index=True)
    return df

print("build_features() 函数定义完成")
print("用法: df = build_features(ds, selected_weather_features=['gust', 't2m', 'cape', ...])")
print("如果不传 selected_weather_features，会用全部 109 个天气变量")

In [ ]:
# Task C: 试运行 — 用筛选后的 top 特征构造
# 先用 B1 得到的 top 15 特征测试一下（跑完 B1 后这里才有值）
try:
    top15_features = list(corr_series.head(15).index)
    print(f"使用 top 15 天气特征: {top15_features}\n")
    
    df = build_features(ds, selected_weather_features=top15_features)
    
    print(f"构造后的 DataFrame:")
    print(f"  Shape: {df.shape}  ({df.shape[0]:,} 行 x {df.shape[1]} 列)")
    print(f"  列名: {list(df.columns)}")
    print(f"\n  前几行预览:")
    print(df.head(3).to_string())
    
    # 检查 NaN（lag 特征在前几行会有 NaN，这是正常的）
    nan_counts = df.isnull().sum()
    print(f"\n  有 NaN 的列 (lag/rolling 的前几行会有，正常):")
    print(nan_counts[nan_counts > 0])
except NameError:
    print("请先运行 Task B 的代码得到 corr_series")

---
# Task D: Baseline 复现 & 评估框架

**目标**：跑通 baseline，记录 RMSE，搭建统一评估函数。

## D1: 评估函数

统一的评估函数，所有模型都用这个来算分，方便对比。

In [ ]:
# D1: 统一评估函数
def evaluate_model(y_true, y_pred, locations, model_name="Model"):
    """
    评估模型预测结果。
    
    参数:
        y_true: (T, L) 真实停电数，T=时间步数, L=县数量
        y_pred: (T, L) 预测停电数
        locations: 县名列表
        model_name: 模型名称（用于打印）
    
    返回:
        overall_rmse: 全局平均 RMSE
        county_rmses: dict, {county: rmse}
    """
    county_rmses = {}
    for li, loc in enumerate(locations):
        diff = y_true[:, li] - y_pred[:, li]
        county_rmses[loc] = np.sqrt(np.mean(diff**2))
    
    overall_rmse = np.mean(list(county_rmses.values()))
    
    print(f"=== {model_name} ===")
    print(f"  Overall RMSE (县均): {overall_rmse:.4f}")
    
    # Top 5 最差的县
    sorted_counties = sorted(county_rmses.items(), key=lambda x: x[1], reverse=True)
    print(f"  RMSE 最高的 5 个县:")
    for loc, rmse in sorted_counties[:5]:
        print(f"    {loc}: {rmse:.4f}")
    
    return overall_rmse, county_rmses


def compare_models(results_dict):
    """
    对比多个模型的结果。
    
    参数:
        results_dict: {model_name: overall_rmse}
    """
    print("\n" + "="*50)
    print("模型对比")
    print("="*50)
    df = pd.DataFrame([
        {'Model': name, 'RMSE': rmse} 
        for name, rmse in sorted(results_dict.items(), key=lambda x: x[1])
    ])
    print(df.to_string(index=False))
    print(f"\n当前最优: {df.iloc[0]['Model']} (RMSE={df.iloc[0]['RMSE']:.4f})")

print("evaluate_model() 和 compare_models() 函数定义完成")

## D2: Persistence Baseline

最简单的 baseline：用每个县最后一个观测值作为未来所有时刻的预测。这是"什么都不做"的下限。

In [ ]:
# D2: Persistence baseline + 验证集评估
# 数据划分：前 80% 训练，后 20% 验证（和 demo 一致）
VALIDATION_SPLIT = 0.2
split_idx = int(T * (1 - VALIDATION_SPLIT))

train_outage = outage[:split_idx, :]   # (T_train, L)
val_outage = outage[split_idx:, :]     # (T_val, L)
val_timestamps = timestamps[split_idx:]

print(f"训练集: {split_idx} 小时 ({timestamps[0].date()} ~ {timestamps[split_idx-1].date()})")
print(f"验证集: {T - split_idx} 小时 ({timestamps[split_idx].date()} ~ {timestamps[-1].date()})")

# --- Persistence baseline ---
# 24h: 用训练集最后一个值重复 24 次
# 48h: 用训练集最后一个值重复 48 次

val_24h = val_outage[:24, :]  # 验证集前 24 小时
val_48h = val_outage[:48, :]  # 验证集前 48 小时

last_val = train_outage[-1, :]  # 训练集最后一个时刻的停电数 (L,)

persistence_24h = np.tile(last_val, (24, 1))  # (24, L)
persistence_48h = np.tile(last_val, (48, 1))  # (48, L)

# 评估
results = {}
rmse_24, _ = evaluate_model(val_24h, persistence_24h, locations, "Persistence 24h")
rmse_48, _ = evaluate_model(val_48h, persistence_48h, locations, "Persistence 48h")
results['Persistence_24h'] = rmse_24
results['Persistence_48h'] = rmse_48

# --- Historical Average baseline ---
# 用训练集每个县的全局平均值作为预测
hist_avg = train_outage.mean(axis=0)  # (L,)
hist_avg_24h = np.tile(hist_avg, (24, 1))
hist_avg_48h = np.tile(hist_avg, (48, 1))

print()
rmse_24_ha, _ = evaluate_model(val_24h, hist_avg_24h, locations, "Historical Average 24h")
rmse_48_ha, _ = evaluate_model(val_48h, hist_avg_48h, locations, "Historical Average 48h")
results['HistAvg_24h'] = rmse_24_ha
results['HistAvg_48h'] = rmse_48_ha

print("\n--- 后续: 跑完 demo.ipynb 的 SARIMAX 和 Seq2Seq 后把它们的 RMSE 也填进来 ---")
# results['SARIMAX_24h'] = xxx
# results['SARIMAX_48h'] = xxx
# results['Seq2Seq_24h'] = xxx
# results['Seq2Seq_48h'] = xxx
# compare_models(results)

---
# Phase 1 总结 & 下一步

跑完这个 notebook 后，你应该知道：

1. **数据特点**：停电数据极度稀疏，极端值由天气事件驱动
2. **重要特征**：从 109 个天气变量筛选到了 ~15 个最相关的
3. **特征工程**：`build_features()` 函数可以直接给后续模型用
4. **Baseline 水平**：Persistence / Historical Average / SARIMAX / Seq2Seq 的 RMSE 已知

**Phase 2 的方向**：
- 尝试更强的模型（XGBoost, LightGBM, Transformer 等）
- 用 `build_features()` 的输出作为起点，根据模型特点进一步调整
- 模型集成（Ensemble）
- Part II: 基于预测结果做发电机部署决策